In [1]:
from pathlib import Path
import sys

# Locate the project folder from wherever the notebook is opened
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "model_setup.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_preparation import load_training_bundle
from src.model_setup import make_model_pipeline, shared_cv

# Load shared training dataset
X_train, y_train, metadata = load_training_bundle(ROOT)

# Use the same predefined 5-fold cross-validation
cv = shared_cv(metadata)

print("Training Records:", X_train.shape[0])
print("Number of Features:", X_train.shape[1])
print("CV Folds:", cv.get_n_splits())

print("\nTarget Distribution:")
print(y_train.value_counts())

Training Records: 202795
Number of Features: 21
CV Folds: 5

Target Distribution:
HeartDiseaseorAttack
0    183698
1     19097
Name: count, dtype: int64


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

# Create Decision Tree model
dt_model = make_model_pipeline(
    DecisionTreeClassifier(random_state=42),
    kind="tree"
)

# Define hyperparameter search space
dt_param_distributions = {
    "model__max_depth": [5, 10, 15, 20, 30],
    "model__min_samples_split": [2, 10, 20, 50],
    "model__min_samples_leaf": [1, 10, 30, 50],
    "model__class_weight": [None, "balanced"]
}

print("Decision Tree Optimisation Model Created")
print("\nHyperparameter Search Space:")

for parameter, values in dt_param_distributions.items():
    print(f"{parameter}: {values}")

print("\nTotal Possible Combinations: 160")
print("Cross-Validation Folds:", cv.get_n_splits())
print("Primary Evaluation Metric: Average Precision")

Decision Tree Optimisation Model Created

Hyperparameter Search Space:
model__max_depth: [5, 10, 15, 20, 30]
model__min_samples_split: [2, 10, 20, 50]
model__min_samples_leaf: [1, 10, 30, 50]
model__class_weight: [None, 'balanced']

Total Possible Combinations: 160
Cross-Validation Folds: 5
Primary Evaluation Metric: Average Precision


In [3]:
# Configure RandomizedSearchCV
dt_search = RandomizedSearchCV(
    estimator=dt_model,
    param_distributions=dt_param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1,
    refit=True,
    return_train_score=True
)

# Run hyperparameter optimisation
dt_search.fit(X_train, y_train)

# Display best results
print("\nDecision Tree Optimisation Completed")

print("\nBest Hyperparameters:")
for parameter, value in dt_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest Cross-Validation Average Precision: "
    f"{dt_search.best_score_:.4f}"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Decision Tree Optimisation Completed

Best Hyperparameters:
model__min_samples_split: 10
model__min_samples_leaf: 50
model__max_depth: 10
model__class_weight: None

Best Cross-Validation Average Precision: 0.3407


In [4]:
import pandas as pd
from sklearn.model_selection import cross_validate

# Get the best Decision Tree model
dt_optimised_model = dt_search.best_estimator_

# Define evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "Average_Precision": "average_precision"
}

# Evaluate optimised Decision Tree using 5-fold CV
dt_optimised_cv = cross_validate(
    estimator=dt_optimised_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

# Summarise validation results
dt_optimised_summary = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean": [
        dt_optimised_cv[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Std": [
        dt_optimised_cv[f"test_{metric}"].std()
        for metric in scoring
    ]
})

print("Optimised Decision Tree - 5-Fold CV Results")
display(dt_optimised_summary.round(4))

Optimised Decision Tree - 5-Fold CV Results


,Metric,Mean,Std
0,Accuracy,0.9066,0.0007
1,Precision,0.5233,0.0097
2,Recall,0.0889,0.0081
3,F1,0.1518,0.0119
4,ROC_AUC,0.8340,0.0038
5,Average_Precision,0.3407,0.0060


In [ ]:
import pandas as pd

# Load Decision Tree baseline results
baseline_path = (
    ROOT / "results" / "member2" /
    "decision_tree_baseline.csv"
)
dt_baseline = pd.read_csv(baseline_path)

# Prepare baseline and optimised results
baseline = dt_baseline.set_index("Metric")
optimised = dt_optimised_summary.set_index("Metric")

# Compare mean and standard deviation
dt_comparison = pd.DataFrame({
    "Metric": optimised.index,
    "Baseline Mean": baseline["Mean Validation Score"].reindex(optimised.index).values,
    "Baseline Std": baseline["Standard Deviation"].reindex(optimised.index).values,
    "Optimised Mean": optimised["Mean"].values,
    "Optimised Std": optimised["Std"].values
})

# Calculate mean performance change
dt_comparison["Mean Change"] = (
    dt_comparison["Optimised Mean"] - dt_comparison["Baseline Mean"]
)

print("Decision Tree - Baseline vs Optimised")
display(dt_comparison.round(4))

Decision Tree - Baseline vs Optimised


,Metric,Baseline Mean,Baseline Std,Optimised Mean,Optimised Std,Mean Change
0,Accuracy,0.8521,0.0007,0.9066,0.0007,0.0544
1,Precision,0.2459,0.0032,0.5233,0.0097,0.2774
2,Recall,0.2758,0.0059,0.0889,0.0081,-0.1869
3,F1,0.2600,0.0044,0.1518,0.0119,-0.1082
4,ROC_AUC,0.5962,0.0030,0.8340,0.0038,0.2378
5,Average_Precision,0.1379,0.0022,0.3407,0.0060,0.2028


In [6]:
# Compare optimised training and validation performance
dt_optimised_overfitting = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "ROC_AUC", "Average_Precision"],
    "Training Score": [
        dt_optimised_cv[f"train_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ],
    "Validation Score": [
        dt_optimised_cv[f"test_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ]
})

# Calculate performance gaps
dt_optimised_overfitting["Gap"] = (
    dt_optimised_overfitting["Training Score"]
    - dt_optimised_overfitting["Validation Score"]
)

print("Optimised Decision Tree - Training vs Validation")
display(dt_optimised_overfitting.round(4))

Optimised Decision Tree - Training vs Validation


,Metric,Training Score,Validation Score,Gap
0,Accuracy,0.9091,0.9066,0.0025
1,F1,0.1764,0.1518,0.0246
2,ROC_AUC,0.8568,0.8340,0.0228
3,Average_Precision,0.3924,0.3407,0.0517


In [ ]:
import pandas as pd

# Load baseline tree complexity
baseline_path = (
    ROOT / "results" / "member2" /
    "decision_tree_complexity.csv"
)
dt_baseline_complexity = pd.read_csv(baseline_path)

# Get the optimised Decision Tree
trained_tree = dt_search.best_estimator_.named_steps["model"]

# Compare baseline and optimised tree complexity
dt_complexity_comparison = pd.DataFrame({
    "Structure": [
        "Tree Depth",
        "Leaf Nodes",
        "Total Nodes"
    ],
    "Baseline": [
        dt_baseline_complexity["Tree Depth"].iloc[0],
        dt_baseline_complexity["Leaf Nodes"].iloc[0],
        dt_baseline_complexity["Total Nodes"].iloc[0]
    ],
    "Optimised": [
        trained_tree.get_depth(),
        trained_tree.get_n_leaves(),
        trained_tree.tree_.node_count
    ]
})

# Calculate complexity change
dt_complexity_comparison["Change"] = (
    dt_complexity_comparison["Optimised"] - dt_complexity_comparison["Baseline"]
)

print("Decision Tree - Baseline vs Optimised Complexity")
display(dt_complexity_comparison)

Decision Tree - Baseline vs Optimised Complexity


,Structure,Baseline,Optimised,Change
0,Tree Depth,43,10,-33
1,Leaf Nodes,27795,564,-27231
2,Total Nodes,55589,1127,-54462


In [ ]:
import pandas as pd

# Use the existing results folder
results_dir = ROOT / "results" / "member2"
results_dir.mkdir(parents=True, exist_ok=True)

# Save optimised evaluation results
dt_optimised_summary.to_csv(
    results_dir / "decision_tree_optimised.csv",
    index=False
)

# Save baseline vs optimised performance
dt_comparison.to_csv(
    results_dir / "decision_tree_optimisation_comparison.csv",
    index=False
)

# Save training vs validation analysis
dt_optimised_overfitting.to_csv(
    results_dir / "decision_tree_optimised_overfitting.csv",
    index=False
)

# Save tree complexity comparison
dt_complexity_comparison.to_csv(
    results_dir / "decision_tree_optimised_complexity.csv",
    index=False
)

# Save best hyperparameters
dt_best_params = pd.DataFrame(
    list(dt_search.best_params_.items()),
    columns=["Hyperparameter", "Best Value"]
)

dt_best_params.to_csv(
    results_dir / "decision_tree_best_parameters.csv",
    index=False
)

print("All Decision Tree Optimisation Results Saved Successfully")

All Decision Tree Optimisation Results Saved Successfully
